# Post-training and Reasoning: From Pretrained Model to Clinical Assistant

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week05_post_training.ipynb)

*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.*

**BME 6938 Advanced Biomedical AI · Week 5 · Reference notebook (ungraded)**

This notebook follows the Week 5 lecture and its four clips. It uses one small open model, SmolLM2-360M, in two versions: the
pretrained *base* model and the *post-trained* model. It reproduces every number from the clips and then trains small LoRA
adapters on a CPU. The running example is the clinical note *"The patient denies chest pain"* and the question
*"Does the patient report chest pain?"*, which the post-trained model gets wrong.

| Section | Clip | What you will see |
|---|---|---|
| 1 | From pretrained model to clinical assistant | Base and post-trained outputs, the chat template, the loss counted only on answer tokens |
| 2 | LoRA | The memory cost of full fine-tuning, LoRA written from scratch, a short fine-tuning run |
| 3 | Preference tuning | The DPO loss, the model's real log-probabilities, and how long a DPO run it takes to fix the negation error |
| 4 | Reasoning models | Rule-based rewards, GRPO group advantages, and what asking a 360M model to reason actually does |

**How to run it**
- **Google Colab:** *Runtime > Run all*. A GPU (*Runtime > Change runtime type > T4 GPU*) makes it faster, but CPU works.
- **HiPerGator or your own computer:** open it in Jupyter with a PyTorch kernel. On a CPU the whole notebook takes about three minutes,
  including the optional cell at the end.
- The first run downloads SmolLM2-360M and SmolLM2-360M-Instruct (under 1 GB each, Apache 2.0, no account needed).

**Credits.** The LoRA layer, the masked fine-tuning loss, and the DPO loss follow the approach of Sebastian Raschka's
[LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (chapter 7, appendix E, and the DPO bonus notebook), and the
GRPO advantage follows his [reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch). Both repositories are
licensed under Apache 2.0. The code here is rewritten for this course and its clinical examples.

**How to read it:** read the text above each cell, predict what the cell will print, then run it. Nothing here is graded.

## 0. Setup

In [ ]:
# Install the libraries this notebook needs if they are missing (Colab already has them).
import importlib, subprocess, sys
for package in ["torch", "transformers", "accelerate"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import gc, math, re, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)                                            # fixed seed so random numbers are the same every run
device = "cuda" if torch.cuda.is_available() else "cpu"         # use a GPU if there is one
dtype = torch.float32                                           # 32-bit everywhere, so the outputs match the clips
print("PyTorch", torch.__version__, "| device:", device)

A few helpers used throughout. `chat` wraps a question in the model's chat template and generates an answer with greedy
decoding (always the most likely next token), so the output is the same every run.

In [ ]:
NOTE_Q = "Note: The patient denies chest pain.\nQuestion: Does the patient report chest pain?"   # the running example

def load(name):
    # Load a tokenizer and model, move the model to the device, and switch off dropout.
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(name, dtype=dtype).to(device).eval()
    return tok, mdl

def continue_text(tok, mdl, text, n=30):
    # Greedy continuation of raw text, with no chat template.
    ids = tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = mdl.generate(**ids, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True)

REASON = "\n\nThink step by step first, then give the answer on a final line."

def prompt_ids(tok, question, reason=False):
    # The question wrapped in the chat template, ending where the assistant's answer starts.
    # SmolLM2 has no thinking switch (section 4.3), so reasoning is asked for in the prompt.
    text = tok.apply_chat_template([{"role": "user", "content": question + (REASON if reason else "")}],
                                   tokenize=False, add_generation_prompt=True)
    return tok(text, return_tensors="pt").input_ids.to(device)

def chat(tok, mdl, question, reason=False, n=60, **sampling):
    # Answer a question through the chat template; returns the text and the number of generated tokens.
    ids = prompt_ids(tok, question, reason)
    with torch.no_grad():
        out = mdl.generate(ids, max_new_tokens=n, do_sample=bool(sampling), **sampling)
    new = out[0, ids.shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), len(new)

## 1. From pretrained model to clinical assistant  *(clip 1)*

### 1.1 The base model only continues text
The base model was trained only to predict the next token. Give it the start of a discharge summary and it keeps
writing. At this size it writes the same sentence twice, which is a reminder that a base model continues text without
any sense of having answered.

In [ ]:
tok_base, base = load("HuggingFaceTB/SmolLM2-360M")
print(continue_text(tok_base, base, "Discharge summary: The patient denies chest pain."))

Now give it our note and question, as plain text. It does not answer like an assistant: it treats the text as a quiz and
keeps writing more questions (clip 1, segment 2).

In [ ]:
print(continue_text(tok_base, base, NOTE_Q))

In [ ]:
del base; _ = gc.collect()                                      # free memory before loading the second model

### 1.2 The post-trained model answers, but wrongly
The post-trained model expects a chat template. Print it once: the special markers `<|im_start|>` and `<|im_end|>` mark where
each turn begins and ends. SmolLM2 also adds a system turn of its own that we never wrote, which is why the printout
begins by telling the model its name. Every one of those tokens is part of the prompt, as the next cell shows.

In [ ]:
tok, model = load("HuggingFaceTB/SmolLM2-360M-Instruct")
print(tok.apply_chat_template([{"role": "user", "content": NOTE_Q}], tokenize=False,
                              add_generation_prompt=True))

In [ ]:
answer, n_tokens = chat(tok, model, NOTE_Q)
print(answer)                                                   # clip 1, segment 3: "Yes, the patient reports chest pain."

### 1.3 Supervised fine-tuning counts the loss only on the answer
Supervised fine-tuning uses the ordinary next-token loss, with one change: the loss is counted only on the answer tokens.
In PyTorch this is done by setting the label of every prompt token to `-100`, which `cross_entropy` ignores.

In [ ]:
def sft_example(tok, question, answer):
    # Token ids for prompt + answer, and labels that are -100 (ignored) on every prompt token.
    p = prompt_ids(tok, question)[0]
    a = tok(answer + "<|im_end|>", add_special_tokens=False, return_tensors="pt").input_ids[0].to(device)
    ids = torch.cat([p, a])
    labels = torch.cat([torch.full_like(p, -100), a])           # -100: not counted in the loss
    return ids, labels

ids, labels = sft_example(tok, NOTE_Q, "No. The note says the patient denies chest pain.")
for t, l in zip(ids.tolist(), labels.tolist()):
    print(f"{tok.decode([t])!r:28} {'counted' if l != -100 else 'not counted'}")

Hugging Face models shift the labels by one position and apply this rule for us when we pass `labels`. We check that it equals
the mean next-token loss over the answer tokens only, computed by hand.

In [ ]:
def sft_loss(mdl, ids, labels):
    # Mean next-token loss over the answer tokens: the logits at position i predict token i+1.
    logits = mdl(ids[None]).logits[0]
    return F.cross_entropy(logits[:-1], labels[1:], ignore_index=-100)

with torch.no_grad():
    ours = sft_loss(model, ids, labels)
    theirs = model(ids[None], labels=labels[None]).loss
print(f"by hand: {ours.item():.4f}   Hugging Face: {theirs.item():.4f}")

## 2. LoRA: fine-tuning with a small fraction of the weights  *(clip 2)*

### 2.1 What full fine-tuning costs
With the Adam optimizer in 32-bit floating point, each parameter needs its weight (4 bytes), its gradient (4 bytes), and two
optimizer states (4 bytes each): 16 bytes, before counting activations.

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print(f"SmolLM2-360M: {n_params:,} parameters x 16 bytes = {n_params * 16 / 1e9:.1f} GB")
print(f"A 70B model: 70e9 x 16 bytes = {70e9 * 16 / 1e9:,.0f} GB")

### 2.2 LoRA from scratch
LoRA freezes a weight matrix W and adds a low-rank update beside it: `output = W·x + (alpha / r) · B·A·x`.
A has shape r × d_in and B has shape d_out × r. **A starts small and random; B starts at zero**, so at the start the update is
zero and the model is unchanged (clip 2, segment 5).

The factor `alpha / r` is not in the clip. It scales the update so that you can change the rank r without retuning the
learning rate. Here r = 8 and alpha = 16, so the factor is 2.

In [ ]:
class LoRALinear(nn.Module):
    # A frozen linear layer plus a trainable low-rank update B·A.
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base                                            # the original layer, frozen
        self.A = nn.Parameter(torch.randn(r, base.in_features, device=base.weight.device) * 0.01)   # small random
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=base.weight.device))         # zero
        self.scale = alpha / r                                      # the LoRA scaling factor
    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

PROJ = ("q_proj", "k_proj", "v_proj", "o_proj")                     # the four attention projections

def add_lora(mdl, r=8, alpha=16):
    # Freeze every weight, then wrap the four attention projections of every layer with LoRA.
    for p in mdl.parameters():
        p.requires_grad = False
    for layer in mdl.model.layers:
        for name in PROJ:
            setattr(layer.self_attn, name, LoRALinear(getattr(layer.self_attn, name), r, alpha))
    return [p for p in mdl.parameters() if p.requires_grad]        # only A and B are trainable

def remove_lora(mdl):
    # Put the original layers back, discarding the adapters.
    for layer in mdl.model.layers:
        for name in PROJ:
            m = getattr(layer.self_attn, name)
            if isinstance(m, LoRALinear):
                setattr(layer.self_attn, name, m.base)

The shapes and counts from the clip: one query projection, then all 32 layers.

In [ ]:
q = model.model.layers[0].self_attn.q_proj
print("query projection W:", tuple(q.weight.shape), "=", f"{q.weight.numel():,}", "weights")
print(f"B: {q.out_features:,} x 8 + A: 8 x {q.in_features:,} =", f"{q.out_features * 8 + 8 * q.in_features:,}",
      f"({(q.out_features * 8 + 8 * q.in_features) / q.weight.numel():.1%} of W)")

with torch.no_grad():
    before = model(ids[None]).logits                              # output of the original model
trainable = add_lora(model)
n_lora = sum(p.numel() for p in trainable)
print(f"LoRA weights in all 32 layers: {n_lora:,} = {n_lora / n_params:.2%} of the model")
print(f"adapter file in 16-bit: {n_lora * 2 / 1e6:.1f} MB")
with torch.no_grad():
    after = model(ids[None]).logits
print("output unchanged at the start (B = 0):", torch.allclose(before, after))

### 2.2b Watch rank buy back the update

LoRA's claim is that the change a fine-tune makes to a weight matrix is low rank, so a product of two thin matrices can
stand in for it. The animation tests that claim on a synthetic update: it sweeps the rank from 1 upward, and at each rank
shows the best possible approximation of that rank beside the update it is approximating.

The left pair is the target and the reconstruction. The right panel is the error, and the shape of that curve is the whole
argument: it falls steeply over the first few ranks and then flattens, so most of the update lives in a handful of
directions. The rank where it flattens is the rank worth paying for.

The approximation here comes from an SVD, which gives the best reconstruction of each rank and so an upper bound on what
LoRA can learn. A trained LoRA gets close to this curve, not above it.

In [ ]:
# An animation, drawn by the code below. It renders with to_jshtml, which needs no
# ffmpeg and plays in Colab, in Jupyter, and on nbviewer. Press play, or drag the
# slider to step through it frame by frame.
import numpy as np                                           # arrays
import matplotlib.pyplot as plt                              # figures
from matplotlib.animation import FuncAnimation               # the animation itself
from IPython.display import HTML                             # shows it in the notebook

UF, ORANGE, GREY = "#0021A5", "#FA4616", "#6B7280"           # the course colours
plt.rcParams["figure.dpi"] = 78                              # keeps the rendered animation light

D, R_TRUE, MAXR = 40, 4, 16                                  # matrix size, true rank, highest rank we try
rng = np.random.default_rng(0)
# A synthetic weight update that is genuinely low rank, plus a little full-rank noise,
# which is what a real fine-tune looks like: mostly a few directions, not exactly.
dW = (rng.normal(0, 1, (D, R_TRUE)) @ rng.normal(0, 1, (R_TRUE, D))) / np.sqrt(R_TRUE)
dW += rng.normal(0, 0.18, (D, D))
U, S, Vt = np.linalg.svd(dW)                                 # the best rank-r approximation for every r at once
err = [np.linalg.norm(dW - (U[:, :r] * S[:r]) @ Vt[:r]) / np.linalg.norm(dW)
       for r in range(1, MAXR + 1)]                          # relative error at each rank
lim = np.abs(dW).max()

HOLD = 3                                                     # frames per rank, so the eye can keep up
fig, (a_t, a_r, a_e) = plt.subplots(1, 3, figsize=(9.6, 3.2),
                                    gridspec_kw={"width_ratios": [1, 1, 1.25]})
fig.subplots_adjust(wspace=0.35)

def draw(f):
    r = f // HOLD + 1                                        # the rank being shown
    approx = (U[:, :r] * S[:r]) @ Vt[:r]                     # B @ A, in LoRA's notation
    for ax in (a_t, a_r, a_e):
        ax.clear()
    a_t.imshow(dW, cmap="RdBu_r", vmin=-lim, vmax=lim); a_t.set_title("the update, dW", fontsize=11)
    a_r.imshow(approx, cmap="RdBu_r", vmin=-lim, vmax=lim)
    a_r.set_title(f"rank {r}: B @ A", fontsize=11)
    for ax in (a_t, a_r):
        ax.set_xticks([]); ax.set_yticks([])
    a_e.plot(range(1, MAXR + 1), err, color="#C7CDD1", lw=1.6)
    a_e.plot(range(1, r + 1), err[:r], color=UF, lw=2.4)
    a_e.scatter([r], [err[r - 1]], color=ORANGE, zorder=3, s=36)
    a_e.set_xlim(1, MAXR); a_e.set_ylim(0, 1.02)
    a_e.set_xlabel("rank r", fontsize=10); a_e.set_ylabel("relative error", fontsize=10)
    a_e.spines[["top", "right"]].set_visible(False)
    params = 2 * D * r                                       # the parameters LoRA would train at this rank
    a_e.set_title(f"{params:,} trained, {D * D:,} full  ({100 * params / D**2:.0f}%)",
                  loc="right", fontsize=10, color=GREY)
    return []

ani = FuncAnimation(fig, draw, frames=MAXR * HOLD, interval=150)
plt.close(fig)
HTML(ani.to_jshtml(fps=7))

### 2.3 A short fine-tuning run
We fine-tune only the LoRA weights on 12 synthetic note-and-question pairs: six symptoms, each once with "denies" and once
with "reports". Headache and leg swelling are held out, to see whether the model learned the pattern or memorized the
examples. On a CPU this takes about ten seconds.

In [ ]:
def note_qa(verb, symptom):
    # One synthetic training pair: a one-line note, a question, and the correct answer.
    question = f"Note: The patient {verb} {symptom}.\nQuestion: Does the patient report {symptom}?"
    answer = f"{'No' if verb == 'denies' else 'Yes'}. The note says the patient {verb} {symptom}."
    return question, answer

TRAIN = ["chest pain", "shortness of breath", "fever", "nausea", "cough", "palpitations"]
sft_data = [sft_example(tok, *note_qa(v, s)) for s in TRAIN for v in ("denies", "reports")]
TEST = [note_qa(v, s)[0] for s in ("chest pain", "headache", "leg swelling") for v in ("denies", "reports")]

def show(questions):
    # Greedy answers of the current model to a list of questions.
    model.eval()
    for qn in questions:
        print(f"{qn.splitlines()[0]:42} -> {chat(tok, model, qn, n=25)[0]}")

print("before fine-tuning:"); show(TEST)

In [ ]:
opt = torch.optim.AdamW(trainable, lr=1e-4)                      # only the LoRA weights are updated
model.train(); t0 = time.time()
for epoch in range(3):
    total = 0.0
    for ids_i, labels_i in sft_data:
        loss = sft_loss(model, ids_i, labels_i)                  # loss on the answer tokens only
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"epoch {epoch + 1}: mean loss {total / len(sft_data):.3f}   ({time.time() - t0:.0f} s)")
print("after fine-tuning:"); show(TEST)

In [ ]:
remove_lora(model); model.eval()                                 # back to the original model for section 3
print(chat(tok, model, NOTE_Q)[0])                               # the original (wrong) answer again

## 3. Preference tuning: RLHF and DPO  *(clip 3)*

### 3.1 The DPO loss
For each answer, the **gain** is its log-probability under the model being trained minus its log-probability under a frozen
copy of the starting model. The DPO loss is

`loss = −log σ( β × (chosen gain − rejected gain) )`

First we reproduce the two numbers from the clip.

In [ ]:
def dpo_loss(chosen_gain, rejected_gain, beta=0.1):
    # -log sigmoid(beta * gap); F.logsigmoid is the numerically stable log(sigmoid(x)).
    return -F.logsigmoid(beta * (chosen_gain - rejected_gain))

print(f"start (0, 0):          {dpo_loss(torch.tensor(0.0), torch.tensor(0.0)):.2f}")   # 0.69
print(f"after training (+4, -6): {dpo_loss(torch.tensor(4.0), torch.tensor(-6.0)):.2f}")  # 0.31

### 3.2 What the model prefers now
The log-probability of an answer is the sum of the log-probabilities of its tokens, given the prompt. The clip's numbers (−21.37 for the chosen answer and −1.12 for the
rejected one) came from this cell with the model the clip used. SmolLM2-360M-Instruct gives about −21.1 and −2.2: a
different pair of numbers, but the same and much larger gap, because the fluent wrong answer is the one the model finds
almost unsurprising.

In [ ]:
CHOSEN = "No. The note says the patient denies chest pain."
REJECTED = "Yes, the patient reports chest pain."

def answer_logprob(mdl, question, answer):
    # Sum of log-probabilities of the answer tokens (including the end marker), given the prompt.
    ids, labels = sft_example(tok, question, answer)
    logp = torch.log_softmax(mdl(ids[None]).logits[0, :-1], dim=-1)    # position i predicts token i+1
    targets = ids[1:]
    keep = labels[1:] != -100                                          # answer tokens only
    return logp[torch.arange(len(targets), device=device), targets][keep].sum()

with torch.no_grad():
    print(f"chosen:   {answer_logprob(model, NOTE_Q, CHOSEN).item():.2f}")
    print(f"rejected: {answer_logprob(model, NOTE_Q, REJECTED).item():.2f}")

### 3.3 A short DPO run
We train fresh LoRA adapters with DPO on four preference pairs. Eight epochs, which is what this notebook ran until
we moved to a 360M model, move the log-probabilities in the right direction but leave the greedy answer unchanged, so
we run 16. Because LoRA leaves the original weights untouched, the
**frozen reference** is simply the same model with its adapters switched off, so we do not need a second copy in memory: we
compute the reference log-probabilities once, before training.

The learning rate matters. With a learning rate 25 times higher, the rejected log-probability collapses, here to about
−74, and the model starts repeating "the note says the note denies the note denies". Pushing the rejected answer down
hard enough damages the writing itself; this is a known way DPO can fail, and you can try it below.

In [ ]:
PAIRS = [(NOTE_Q, CHOSEN, REJECTED)]
for s in ("shortness of breath", "fever"):
    qn, good = note_qa("denies", s)
    PAIRS.append((qn, good, f"Yes, the patient reports {s}."))
qn, good = note_qa("reports", "nausea")                          # one pair where the answer is "Yes", for balance
PAIRS.append((qn, good, "No, the patient does not report nausea."))

with torch.no_grad():                                            # reference log-probabilities, from the starting model
    REF = [(answer_logprob(model, q_, c), answer_logprob(model, q_, r)) for q_, c, r in PAIRS]

trainable = add_lora(model)
opt = torch.optim.AdamW(trainable, lr=2e-5)
model.train(); t0 = time.time()
for epoch in range(16):
    total = 0.0
    for (q_, c, r), (ref_c, ref_r) in zip(PAIRS, REF):
        gain_c = answer_logprob(model, q_, c) - ref_c            # chosen gain
        gain_r = answer_logprob(model, q_, r) - ref_r            # rejected gain
        loss = dpo_loss(gain_c, gain_r)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"epoch {epoch + 1}: loss {total / len(PAIRS):.3f}   last pair: chosen gain {gain_c.item():+.1f}, "
          f"rejected gain {gain_r.item():+.1f}   ({time.time() - t0:.0f} s)")

The loss starts at 0.69, as in the clip, and falls as the chosen gain pulls ahead while the rejected gain barely moves:
DPO is raising the good answer here rather than pushing the bad one down. Watch where the answer flips. Up to about
epoch 8 the model still says "Yes", even though the log-probabilities have already started to separate, which is worth
remembering: a preference loss that is clearly falling does not yet mean the behaviour has changed. Now ask again,
including headache, which was never in the preference data, and a note where the right answer is "Yes".

In [ ]:
show([NOTE_Q, note_qa("denies", "headache")[0], note_qa("reports", "chest pain")[0]])
remove_lora(model); _ = model.eval()                             # back to the original model for section 4

> **A caution for clinical preference data.** Preference pairs that always reward the more cautious answer can teach a model
> to hedge on everything ("further assessment is recommended") instead of answering. Good preference data rewards the answer
> that is *correct and useful*, and includes cases where the confident answer is the right one, like the "reports nausea" pair above.

## 4. Reasoning models  *(clip 4)*

### 4.1 Rule-based rewards
DeepSeek-R1 was trained with rewards that a program can check: an **accuracy reward** (is the final answer correct?) and a
**format reward** (is the reasoning inside `<think>` tags?). Here are both for a multiple-choice question with answer B.

In [ ]:
ECG_Q = ("A 58-year-old man has crushing chest pain. The ECG shows ST elevation in leads II, III, and aVF. "
         "What is the most likely occluded artery?\nA) Left anterior descending\nB) Right coronary artery\n"
         "C) Left circumflex\nD) Left main\nAnswer with the letter.")
CORRECT = "B"

def final_letter(text):
    # The last answer letter A-D written as "B)" or "**B" or "Answer: B" in the text, or None.
    found = re.findall(r"(?:\*\*|Answer:\s*|\b)([A-D])\)", text) or re.findall(r"Answer:\s*\**([A-D])\b", text)
    return found[-1] if found else None

def accuracy_reward(text):
    return 1.0 if final_letter(text) == CORRECT else 0.0

def format_reward(text):
    return 1.0 if re.search(r"<think>.+?</think>", text, re.S) else 0.0

example = "<think>Inferior leads point to the inferior wall, usually the right coronary artery.</think> Answer: B)"
print("accuracy:", accuracy_reward(example), " format:", format_reward(example))

### 4.2 GRPO: compare each reward with its own group
GRPO samples a group of answers to the same question and scores each one relative to the group:
`advantage = (reward − mean) ÷ std`. The clip uses the **population** standard deviation (divide by n, `unbiased=False`).
With the sample standard deviation (divide by n − 1, PyTorch's default) the same rewards give +1.5 and −0.5.

In [ ]:
def group_advantages(rewards):
    # Group-relative advantage, as in GRPO; the small constant avoids dividing by zero when all rewards are equal.
    r = torch.tensor(rewards, dtype=torch.float32)
    return (r - r.mean()) / (r.std(unbiased=False) + 1e-6)

r = torch.tensor([1.0, 0.0, 0.0, 0.0])                           # the clip: only sample 1 picks B
print("mean", r.mean().item(), " std", round(r.std(unbiased=False).item(), 3))
print("advantages (population std):", [round(a, 2) for a in group_advantages(r.tolist()).tolist()])
print("advantages (sample std):    ", [round(a, 2) for a in ((r - r.mean()) / r.std()).tolist()])

Now a real group. We sample 8 answers from SmolLM2-360M-Instruct, score each with the accuracy reward, and compute the
advantages. This group's numbers will differ from the clip's illustrative four. Read the sampled text, not just the letters: in our
run every sample named a letter, one of the eight picked B, and that one wrote "B) Left Coronary artery", which is the
right letter beside the wrong artery. The accuracy reward cannot see the difference, and a reward a model can satisfy
without being right is a reward it will learn to satisfy that way.

In [ ]:
torch.manual_seed(0)
ids = prompt_ids(tok, ECG_Q)
with torch.no_grad():
    outs = model.generate(ids, max_new_tokens=40, do_sample=True, temperature=1.0, num_return_sequences=8)
group = [tok.decode(o[ids.shape[1]:], skip_special_tokens=True) for o in outs]
rewards = [accuracy_reward(g) for g in group]
for g, rw, adv in zip(group, rewards, group_advantages(rewards).tolist()):
    print(f"letter {final_letter(g) or '-'}  reward {rw:.0f}  advantage {adv:+.2f}   {g[:60]!r}")

### 4.3 Thinking at test time
SmolLM2-360M has no thinking switch, so we ask for the reasoning in the prompt instead: the same question with "Think
step by step first" added. The clip's claim (segment 6) is that reasoning at test time fixes the negation error at the
cost of many more tokens. The cost arrives; the fix does not. The direct answer is wrong in 9 tokens, and the reasoning
version spends all 600 tokens we allow it on a loop of numbered steps that restate the note and never reach an answer.
That failure is worth as much as a success would have been: test-time reasoning is a property of models trained for it,
not something a prompt adds to any model.

In [ ]:
for reason in (False, True):
    t0 = time.time()
    text, n = chat(tok, model, NOTE_Q, reason=reason, n=600)
    print(f"{'reason first' if reason else 'direct      '}: {n} tokens, {time.time() - t0:.1f} s")
    print(text, "\n")

### 4.4 A fluent but wrong rationale
This is the clip's point in segment 8, and it survives the change of model intact. Asked directly, the model answers D,
Left main, which is wrong. Asked to reason first, it invents an ECG finding that was never in the question (ST elevation
in leads I, II, and aVF), argues its way to the left anterior descending, and then answers D anyway, so the rationale
does not even agree with the letter beside it. Read it: it is fluent, it is confident, it is laid out in tidy numbered
steps, and it is wrong from the second line. A rationale is not evidence.

In [ ]:
print("direct:", chat(tok, model, ECG_Q, n=60)[0], "\n")
text, n = chat(tok, model, ECG_Q, reason=True, n=300)
print(f"reasoning first, {n} tokens:\n{text}")

## Recap

| Clip | Code in this notebook |
|---|---|
| 1. From pretrained model to clinical assistant | Base and post-trained outputs, the chat template, `sft_example` and `sft_loss` |
| 2. LoRA | The 16-bytes-per-parameter cost, `LoRALinear`, `add_lora`, a short fine-tuning run |
| 3. Preference tuning | `dpo_loss`, `answer_logprob`, a short DPO run with LoRA |
| 4. Reasoning models | `accuracy_reward`, `format_reward`, `group_advantages`, direct and reasoned answers |

> **Try it (optional):** in section 2.3, change the rank to `r=2` or `r=32` and compare the trainable count and the results.
> In section 3.3, set the learning rate to `5e-4` and watch both gains fall. In section 4.2, sample a group of 16, as DeepSeek-R1 did.